# Rule-Based Expert System for Medical Decision Support - Test file
This notebook imports `RuleAlzheimer`, `questions`, `answer`, `contradictions`, `goals`,  `symptoms_dictionary` and `InferenceEngine` directly from  existing `.py` modules within a shared folder and runs **6 visual end-to-end test scenarios** comparing **Expected vs. Actual** outputs.


In [11]:
# Handling Imports
import copy
from unittest.mock import patch

from inferenceengine import InferenceEngine 
print(f"Successfully imported Inference Engine from 'inferenceengine.py'.")

Successfully imported Inference Engine from 'inferenceengine.py'.


**For Alzheimer**

In [12]:

from alzheimer import goals,questions,answer,contradictions,RuleAlzheimer,symptoms_dictionary # Alzheimer's KB
print(f"Successfully imported Knowledge Base from 'alzheimer.py'.")

Successfully imported Knowledge Base from 'alzheimer.py'.


In [13]:

#  Expected vs. Actual Comparison Helper Function

test_summary_records = []

def run_visual_test(
    test_name: str,
    description: str,
    mode: str, # either forward or backward chaining
    simulated_inputs: list, # list of answers to each question
    expected: dict, # dictionary of answers
    run_explain: bool = True # explain answers
):
    """
    Runs scenario with mocked user inputs and renders Expected vs. Actual output.
    """
    engine = InferenceEngine(
        questions=questions,
        answers=copy.deepcopy(answer),
        rule_class=RuleAlzheimer,
        contradictions=contradictions,
        goals=goals,
        symptoms_dictionary=symptoms_dictionary 
    )
    print(f"RUNNING: {test_name}")
   
    with patch("builtins.input", side_effect=simulated_inputs):
        if mode == "forward":
            engine.forward_chaining()
        elif mode == "backward":
            engine.backward_chaining()
        else:
            raise ValueError("mode must be 'forward' or 'backward'")

    if run_explain:
        print("\n--- Rule Audit Explanation ---")
        engine.explain_rules()

    # Gather actual state
    triggered = engine.kb.print_rules()
    actual_conclusion = triggered[0][0] if triggered else "None"
    actual_confidence = triggered[0][1] if triggered else 0.0
    actual_yes = sum(1 for v in engine.answers.values() if v == "yes")
    actual_no = sum(1 for v in engine.answers.values() if v == "no")
    actual_dk = sum(1 for v in engine.answers.values() if v == "don't know")
    actual_unresolved = len(engine.unresolved_contradictions)

    actual = {
        "conclusion": actual_conclusion,
        "confidence": actual_confidence,
        "yes_count": actual_yes,
        "no_count": actual_no,
        "dk_count": actual_dk,
        "unresolved_contradictions": actual_unresolved,
    }

    checks = [
        ("Conclusion", expected["conclusion"], actual["conclusion"]),
        ("Confidence Score", expected["confidence"], actual["confidence"]),
        ("Confirmed 'yes' Signs", expected["yes_count"], actual["yes_count"]),
        ("Confirmed 'no' Signs", expected["no_count"], actual["no_count"]),
        ("Unknown ('don't know')", expected["dk_count"], actual["dk_count"]),
        ("Unresolved Contradictions", expected["unresolved_contradictions"], actual["unresolved_contradictions"]),
    ]

    all_passed = all(exp == act for _, exp, act in checks)

    test_summary_records.append({
        "name": test_name,
        "passed": all_passed,
        "confidence": actual_confidence,
        "yes": actual_yes,
        "no": actual_no,
        "dk": actual_dk
    })

In [14]:

# TEST 1:Forward Chaining — 0 Warning Signs

run_visual_test(
    test_name="Test 1: 0 Signs",
    description="User answers 'no' to all 10 warning signs. Expects 'No warning signs reported' with 0.0 confidence.",
    mode="forward",
    simulated_inputs=["no"] * 10,
    expected={
        "conclusion": "No warning signs reported",
        "confidence": 0.0,
        "yes_count": 0,
        "no_count": 10,
        "dk_count": 0,
        "unresolved_contradictions": 0,
    }
)

RUNNING: Test 1: 0 Signs
Rule-Based Expert System for Medical Decision Support.
All rules are illustrative and are not intended for actual diagnosis
+------------------+--------------------+------------------------+-------------------------+-----------------+------------------------------+
| Conclusion       |   Confidence_score | Recommendation         | Rationale               | Source          | Rule as text                 |
+==================+====================+========================+=========================+=================+==============================+
| No warning signs |                  0 | No warning signs were  | None of the ten warning | Team assumption | If all warning signs = no or |
| reported         |                    | reported. Seek         | signs were reported.    |                 | don't know, THEN no warning  |
|                  |                    | professional advice if |                         |                 | signs reported               |

In [15]:

# TEST 2: Single Base Rule Trigger (Forward Chaining — 1 Warning Sign)

run_visual_test(
    test_name="Test 2: Single Base Rule (Language Sign Only)",
    description="User answers 'yes' only to 'language' . Expects Base Rule 8 ('Possible communication concern', confidence 0.40).",
    mode="forward",
    simulated_inputs=[
        "no",   
        "no",   
        "no",  
        "yes",  # language
        "no", "no", "no", "no", "no", "no"
    ],
    expected={
        "conclusion": "Possible communication concern",
        "confidence": 0.4,
        "yes_count": 1,
        "no_count": 9,
        "dk_count": 0,
        "unresolved_contradictions": 0,
    }
)

RUNNING: Test 2: Single Base Rule (Language Sign Only)
Rule-Based Expert System for Medical Decision Support.
All rules are illustrative and are not intended for actual diagnosis
+------------------------+--------------------+----------------------+-----------------------------+-----------------+---------------------------------+
| Conclusion             |   Confidence_score | Recommendation       | Rationale                   | Source          | Rule as text                    |
+========================+====================+======================+=============================+=================+=================================+
| Possible communication |                0.4 | Consult a healthcare | Difficulty finding words or | Team assumption | If language = yes THEN Possible |
| concern                |                    | professional.        | following conversations may |                 | communication concern           |
|                        |                    |         

In [16]:
# TEST 3: Input Validation, Contradiction Resolution & Follow-Up Questions

run_visual_test(
    test_name="Test 3: Contradiction Review and Follow-Up Clarification (3 Signs)",
    description="Tests 'example' and invalid input prompts, resolves (memory_loss, misplacing) contradiction, clarifies 1 of 2 'don't know' follow-ups to 'yes' (3 total signs = 0.70 * 9/10 = 0.63 confidence).",
    mode="forward",
    simulated_inputs=[
        "example",     # Q1: memory_loss -> prints examples
        "maybe",       # Q1: invalid input -> prints error
        "no",          # Q1: memory_loss = no
        "don't know",  # Q2: confusion_time_place = don't know
        "no",          # Q3: familiar_tasks = no
        "no",          # Q4: language = no
        "yes",         # Q5: misplacing = yes (triggers contradiction with memory_loss!)
        "don't know",  # Q6: judgment = don't know
        "no", "no", "no", "no",  # Q7-Q10 = no
        # Contradiction review prompt for (memory_loss, misplacing):
        "yes",         # Review answers? -> yes
        "yes",         # Re-answer memory_loss -> updated to yes
        "yes",         # Re-answer misplacing -> kept as yes
        # Follow-up prompts for the two 'don't know' answers:
        "yes",         # Follow-up confusion_time_place -> updated to yes (now 3 'yes' signs!)
        "no"           # Follow-up judgment -> remains don't know
    ],
    expected={
        "conclusion": "Possible early Alzheimer's indicators",
        "confidence": 0.63,
        "yes_count": 3,
        "no_count": 6,
        "dk_count": 1,
        "unresolved_contradictions": 0,
    }
)

RUNNING: Test 3: Contradiction Review and Follow-Up Clarification (3 Signs)
['Forgetting recently learned information', 'Repeatedly asking the same question', 'Forgetting important dates or events', 'Repeating statements']
Answer must be either 'yes','no' or 'don't know'. Type 'example' to see some sample cases
Possible Inconsistence detected between symptoms  Memory loss  and  Misplacing objects

Follow-up question for Confusion Time Place
Answer updated to yes.

Follow-up question for Judgment
No change made. Answer remains don't know.
Rule-Based Expert System for Medical Decision Support.
All rules are illustrative and are not intended for actual diagnosis
+------------------------+--------------------+----------------------+-----------------------------+-----------------+----------------------------------+
| Conclusion             |   Confidence_score | Recommendation       | Rationale                   | Source          | Rule as text                     |
+=======================

In [17]:

# TEST 4: Unresolved Contradiction Flagging (5 Warning Signs)

run_visual_test(
    test_name="Test 4: Unresolved Contradiction Warning (5 Signs)",
    description="User answers 'yes' to 5 signs including familiar_tasks='yes' and planning='no', then declines review ('no'). Expects Moderate warning signs (0.80) + 1 unresolved contradiction.",
    mode="forward",
    simulated_inputs=[
        "yes",  # memory_loss
        "yes",  # confusion_time_place
        "yes",  # familiar_tasks (contradicts planning='no')
        "yes",  # language
        "yes",  # misplacing
        "no",   # judgment
        "no",   # mood_personality
        "no",   # withdrawal
        "no",   # planning
        "no",   # visual_spatial
        # Contradiction review prompt for (planning, familiar_tasks):
        "no"    # Decline review -> recorded in unresolved_contradictions
    ],
    expected={
        "conclusion": "Moderate Alzheimer's warning signs",
        "confidence": 0.8,
        "yes_count": 5,
        "no_count": 5,
        "dk_count": 0,
        "unresolved_contradictions": 1,
    }
)

RUNNING: Test 4: Unresolved Contradiction Warning (5 Signs)
Possible Inconsistence detected between symptoms  Difficulty planning or solving problems  and  Difficulty completing familiar tasks
Rule-Based Expert System for Medical Decision Support.
All rules are illustrative and are not intended for actual diagnosis
+----------------------+--------------------+--------------------------+------------------------------+-----------------+-----------------------------+
| Conclusion           |   Confidence_score | Recommendation           | Rationale                    | Source          | Rule as text                |
+======================+====================+==========================+==============================+=================+=============================+
| Moderate Alzheimer's |                0.8 | Consult a healthcare     | Five signs affecting         | Team assumption | If there are five signs or  |
| warning signs        |                    | professional for further | mu

In [18]:
# TEST 5: Backward Chaining — Early Goal Satisfaction

run_visual_test(
    test_name="Test 5: Backward Chaining --> Early Success (2-Sign Hypothesis)",
    description="Selects Goal 1 ('Possible cognitive decline', requires 2 'yes'). Answers 'yes' to first 2 questions -> stops immediately and auto-fills remaining 8 as 'don't know' (0.60 * 2/10 = 0.12 confidence).",
    mode="backward",
    simulated_inputs=[
        "1",    # Select Goal 1 (threshold = 2)
        "yes",  # Q1: memory_loss | first condition satisfied
        "yes"   # Q2: confusion_time_place  | second condition satisfied -> Early exit!
    ],
    expected={
        "conclusion": "Possible cognitive decline",
        "confidence": 0.12,
        "yes_count": 2,
        "no_count": 0,
        "dk_count": 8,
        "unresolved_contradictions": 0,
    }
)

RUNNING: Test 5: Backward Chaining --> Early Success (2-Sign Hypothesis)
Initializing backward_chaining...
 Please select a hypothesis from the options below:

Enter 1 to select Possible cognitive decline
Enter 2 to select Possible early Alzheimer's indicators
Enter 3 to select Moderate Alzheimer's warning signs
Enter 4 to select Strong Alzheimer's warning signs
Sucess, all 2 conditions for Possible cognitive decline hypothesis met!
 
Rule-Based Expert System for Medical Decision Support.
All rules are illustrative and are not intended for actual diagnosis
+--------------------+--------------------+-----------------------+-----------------------------+-----------------+----------------------------------+
| Conclusion         |   Confidence_score | Recommendation        | Rationale                   | Source          | Rule as text                     |
+====================+====================+=======================+=============================+=================+====================

In [19]:
# TEST 6: Backward Chaining — Early Impossibility & Rule Audit

run_visual_test(
    test_name="Test 6: Backward Chaining --> Early Failure & Rule Audit (6-Sign Hypothesis)",
    description="Selects Goal 4 ('Strong Alzheimer's warning signs', requires 6 'yes'). Answers 'no' 5 times -> impossible to reach 6, stops at Q5, auto-fills remaining 5 as 'don't know', and runs explain_rules().",
    mode="backward",
    simulated_inputs=[
        "4",                      # Select Goal 4 (threshold = 6)
        "no", "no", "no", "no", "no"  # 5 'no' answers is greater than max possible 'no's which is 4 -> Early exit!
    ],
    expected={
        "conclusion": "",
        "confidence":"",
        "yes_count": "",
        "no_count": "",
        "dk_count": "",
        "unresolved_contradictions": "",
    }
)

RUNNING: Test 6: Backward Chaining --> Early Failure & Rule Audit (6-Sign Hypothesis)
Initializing backward_chaining...
 Please select a hypothesis from the options below:

Enter 1 to select Possible cognitive decline
Enter 2 to select Possible early Alzheimer's indicators
Enter 3 to select Moderate Alzheimer's warning signs
Enter 4 to select Strong Alzheimer's warning signs
Failure, Could not meet the 6 conditions for Strong Alzheimer's warning signs hypothesis, only 0 were met before stopping!
 

--- Rule Audit Explanation ---
+-----------------+------------------------------+-----------+------------------------------+
| Rule_id         | Conclusion                   | Status    | Reason                       |
+=================+==============================+===========+==============================+
| Compound Rule 1 | Possible cognitive decline   | Cannot be | Pending unknown answers: (0  |
| (2 signs)       |                              | evaluated | 'yes' + 5 'don't know' cou

In [ ]:
%pip install pandas # installing pandas

In [ ]:
# Summary of Alzheimer's tests
import pandas as pd
print("Printing a dataframe of test_name, pass_state, confidence, yes count,no count and don't know counts below:")
df = pd.DataFrame(test_summary_records).rename(columns={ 
                                                        "name": "Test_name",
                                                        "passed": "Pass_state",
                                                        "confidence": "Confidence",
                                                        "yes": "Yes count",
                                                        "no": "No count",
                                                        "dk": "Don't know count"
})
df.set_index("Test_name", inplace=True)
df


Printing a dataframe of test_name, pass_state, confidence, yes count,no count and don't know counts below:


,Pass_state,Confidence,Yes count,No count,Don't know count
Test_name,,,,,
Test 1: 0 Signs,True,0.00,0,10,0
Test 2: Single Base Rule (Language Sign Only),True,0.40,1,9,0
Test 3: Contradiction Review and Follow-Up Clarification (3 Signs),True,0.63,3,6,1
Test 4: Unresolved Contradiction Warning (5 Signs),True,0.80,5,5,0
Test 5: Backward Chaining --> Early Success (2-Sign Hypothesis),True,0.12,2,0,8
Test 6: Backward Chaining --> Early Failure & Rule Audit (6-Sign Hypothesis),False,0.00,0,5,5
